# Session 6: Narrative Layer and Full Pipeline

This notebook turns structured explanation sentences into short narrative summaries. It connects the full pipeline from data preparation and forecasting to fuzzy labels, structured explanations, and final human-readable output.

The goal is to produce a clean end-to-end MVP.


In [1]:
import pandas as pd

explanations = pd.read_csv("../src/data/session05_structured_explanations.csv")
explanations

,Model,MAE,RMSE,MAPE,MPE,DA,MAE_Label,MPE_Label,Combined_Label,Structured_Explanation
0,Naive,781.31,948.15,5.468,5.434,0.00,high error,slight overprediction,"high error, slight overprediction",The model has high error and tends to overpred...
1,Seasonal Naive,130.40,152.60,0.929,0.892,96.55,low error,neutral,"low error, neutral",The model is highly accurate and shows no cons...
2,Linear Regression,483.29,547.70,3.472,-0.015,48.28,medium error,neutral,"medium error, neutral",The model has moderate error but no strong dir...
3,ETS,44.67,57.32,0.320,0.049,96.55,low error,neutral,"low error, neutral",The model is highly accurate and shows no cons...
4,HWES (damped),62.69,79.08,0.448,0.404,96.55,low error,neutral,"low error, neutral",The model is highly accurate and shows no cons...
5,SARIMA,45.66,59.78,0.326,0.142,96.55,low error,neutral,"low error, neutral",The model is highly accurate and shows no cons...
6,Prophet,89.01,112.18,0.637,0.611,96.55,low error,neutral,"low error, neutral",The model is highly accurate and shows no cons...


In [2]:
def narrative_summary(model, mae_label, mpe_label, structured_text):
    if mae_label == "low error" and mpe_label == "neutral":
        return (
            f"{model} is one of the strongest models in the comparison. "
            f"{structured_text} "
            "This suggests the model tracks electricity demand closely with minimal systematic error."
        )
    if mae_label == "low error" and mpe_label == "slight underprediction":
        return (
            f"{model} performs well overall. "
            f"{structured_text} "
            "This suggests the model tracks demand closely while staying slightly conservative."
        )
    if mae_label == "low error" and mpe_label == "slight overprediction":
        return (
            f"{model} performs well overall. "
            f"{structured_text} "
            "The slight tendency to overestimate is small enough not to undermine the model's reliability."
        )
    if mae_label == "medium error" and mpe_label == "neutral":
        return (
            f"{model} shows moderate performance. "
            f"{structured_text} "
            "The error is noticeably larger than the best models, though the forecasts remain unbiased."
        )
    if mae_label == "medium error":
        return (
            f"{model} shows moderate performance. "
            f"{structured_text} "
            "This suggests the model is useful, but still leaves room for improvement."
        )
    if mae_label == "high error" and mpe_label == "slight overprediction":
        return (
            f"{model} is the weakest model in this comparison. "
            f"{structured_text} "
            "This suggests the forecasts are less accurate and tend to run above actual demand."
        )
    if mae_label == "high error":
        return (
            f"{model} shows poor forecasting accuracy. "
            f"{structured_text} "
            "This model serves primarily as a baseline to highlight the gains made by more sophisticated approaches."
        )
    return f"{model} shows {mae_label} and {mpe_label}. {structured_text}"

In [3]:
explanations["Narrative"] = explanations.apply(
    lambda r: narrative_summary(
        r["Model"],
        r["MAE_Label"],
        r["MPE_Label"],
        r["Structured_Explanation"]
    ),
    axis=1
)

explanations[["Model", "Structured_Explanation", "Narrative"]]


,Model,Structured_Explanation,Narrative
0,Naive,The model has high error and tends to overpred...,Naive is the weakest model in this comparison....
1,Seasonal Naive,The model is highly accurate and shows no cons...,Seasonal Naive is one of the strongest models ...
2,Linear Regression,The model has moderate error but no strong dir...,Linear Regression shows moderate performance. ...
3,ETS,The model is highly accurate and shows no cons...,ETS is one of the strongest models in the comp...
4,HWES (damped),The model is highly accurate and shows no cons...,HWES (damped) is one of the strongest models i...
5,SARIMA,The model is highly accurate and shows no cons...,SARIMA is one of the strongest models in the c...
6,Prophet,The model is highly accurate and shows no cons...,Prophet is one of the strongest models in the ...


In [4]:
for _, row in explanations.iterrows():
    print(row["Narrative"])
    print()


Naive is the weakest model in this comparison. The model has high error and tends to overpredict demand. This suggests the forecasts are less accurate and tend to run above actual demand.

Seasonal Naive is one of the strongest models in the comparison. The model is highly accurate and shows no consistent directional bias. This suggests the model tracks electricity demand closely with minimal systematic error.

Linear Regression shows moderate performance. The model has moderate error but no strong directional bias. The error is noticeably larger than the best models, though the forecasts remain unbiased.

ETS is one of the strongest models in the comparison. The model is highly accurate and shows no consistent directional bias. This suggests the model tracks electricity demand closely with minimal systematic error.

HWES (damped) is one of the strongest models in the comparison. The model is highly accurate and shows no consistent directional bias. This suggests the model tracks elect

In [5]:
explanations.to_csv("../src/data/session06_narratives.csv", index=False)